<a href="https://colab.research.google.com/github/SANTHOSHSANMAR/CliffWalking/blob/main/cliff_walking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
CS656 RL Assignment - Part I, Problem 8
Cliff Walking (Sutton & Barto, Example 6.6): Sarsa vs Q-learning
Also answers Exercises 6.11 and 6.12 (see comments at the bottom).
"""
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROWS, COLS = 4, 12
START, GOAL = (3, 0), (3, 11)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
ARROWS = ["^", "v", "<", ">"]
ALPHA, GAMMA, EPS = 0.5, 1.0, 0.1


def step(state, a):
    """Environment dynamics. Reward -1 per step, -100 for cliff (back to start)."""
    r, c = state
    dr, dc = ACTIONS[a]
    r = min(max(r + dr, 0), ROWS - 1)
    c = min(max(c + dc, 0), COLS - 1)
    if r == 3 and 1 <= c <= 10:          # the cliff
        return START, -100, False
    if (r, c) == GOAL:
        return (r, c), -1, True
    return (r, c), -1, False


def eps_greedy(Q, s, rng, eps=EPS):
    if rng.random() < eps:
        return int(rng.integers(4))
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))  # random tie-break


def run(method, episodes=500, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((ROWS, COLS, 4))
    rewards = np.zeros(episodes)
    for ep in range(episodes):
        s = START
        a = eps_greedy(Q, s, rng)
        done = False
        while not done:
            s2, r, done = step(s, a)
            a2 = eps_greedy(Q, s2, rng)
            if method == "sarsa":
                target = r + GAMMA * Q[s2][a2] * (not done)
            else:  # q-learning
                target = r + GAMMA * Q[s2].max() * (not done)
            Q[s][a] += ALPHA * (target - Q[s][a])
            s, a = s2, a2
            rewards[ep] += r
    return Q, rewards


def greedy_path(Q):
    grid = [["." for _ in range(COLS)] for _ in range(ROWS)]
    for c in range(1, 11):
        grid[3][c] = "C"
    s, seen = START, set()
    while s != GOAL and s not in seen:
        seen.add(s)
        a = int(np.argmax(Q[s]))
        grid[s[0]][s[1]] = ARROWS[a]
        s, _, _ = step(s, a)
    grid[3][0], grid[3][11] = "S" if grid[3][0] == "." else grid[3][0], "G"
    return "\n".join(" ".join(row) for row in grid)


if __name__ == "__main__":
    EPISODES, RUNS = 500, 500
    avg = {"sarsa": np.zeros(EPISODES), "qlearning": np.zeros(EPISODES)}
    for m in avg:
        for k in range(RUNS):
            Q, R = run(m, EPISODES, seed=k)
            avg[m] += R / RUNS
        print(f"{m}: mean reward over last 100 episodes = {avg[m][-100:].mean():.1f}")

    # Figure: online performance (like the book graph)
    def smooth(x, w=10):
        return np.convolve(x, np.ones(w) / w, mode="valid")

    plt.figure(figsize=(7, 4.5))
    plt.plot(smooth(avg["sarsa"]), label="Sarsa", color="tab:blue")
    plt.plot(smooth(avg["qlearning"]), label="Q-learning", color="tab:red")
    plt.ylim(-100, -20)
    plt.xlabel("Episodes")
    plt.ylabel("Sum of rewards during episode")
    plt.title(f"Cliff Walking (eps={EPS}, alpha={ALPHA}, avg of {RUNS} runs)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # Fix for FileNotFoundError: ensure directory exists
    output_path = "/mnt/user-data/outputs/cliff_walking_result.png"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, dpi=150)

    # Learned greedy paths from a single long run
    Qs, _ = run("sarsa", 500, seed=1)
    Qq, _ = run("qlearning", 500, seed=1)
    print("\nSarsa greedy path (safe, upper route):\n" + greedy_path(Qs))
    print("\nQ-learning greedy path (optimal, cliff edge):\n" + greedy_path(Qq))

sarsa: mean reward over last 100 episodes = -26.9
qlearning: mean reward over last 100 episodes = -49.7

Sarsa greedy path (safe, upper route):
. . . . . . . > > > > v
> > > > > > > ^ . . . v
^ . . . . . . . . . . v
^ C C C C C C C C C C G

Q-learning greedy path (optimal, cliff edge):
. . . . . . . . . . . .
. . . . . . . . . . . .
> > > > > > > > > > > v
^ C C C C C C C C C C G
